# 07. SBOM Generation & Analysis

## Background

A Software Bill of Materials (SBOM) is a machine-readable inventory of every component in a software artifact — direct dependencies, transitive dependencies, their versions, licenses, and provenance hashes. Think of it as a nutritional label for software.

Two dominant standards exist: **CycloneDX** (OWASP, JSON/XML) and **SPDX** (Linux Foundation, SPDX-TV/JSON/RDF). US Executive Order 14028 (2021) mandated SBOM delivery for software sold to the federal government, accelerating industry adoption. Tools like Syft, cdxgen, and pip-licenses generate SBOMs automatically from package manifests.

## What You'll Learn

- CycloneDX and SPDX SBOM structure and key fields
- Generating an SBOM from a Python project's dependency tree
- ML model SBOMs: capturing training data, base models, and fine-tuning provenance
- License compliance analysis from SBOM data

## Why This Matters

SBOMs are foundational to supply chain security. An SBOM lets you answer 'are we affected?' within minutes of a new CVE dropping, rather than spending days auditing requirements files. For AI/ML systems the stakes are higher — an SBOM for a model should capture not just code dependencies but also training data lineage and model weights provenance.


## CycloneDX SBOM Builder

In [ ]:
import json
import hashlib
import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict
import uuid

@dataclass
class SBOMComponent:
    name: str
    version: str
    purl: str           # Package URL: pkg:pypi/requests@2.28.0
    licenses: List[str]
    hashes: Dict[str, str] = field(default_factory=dict)
    description: str = ''
    component_type: str = 'library'

class CycloneDXGenerator:
    def __init__(self, project_name: str, version: str):
        self.project_name = project_name
        self.version = version
        self.components: List[SBOMComponent] = []

    def add_component(self, component: SBOMComponent):
        self.components.append(component)

    def _make_hash(self, name: str, version: str) -> str:
        return hashlib.sha256(f'{name}=={version}'.encode()).hexdigest()[:16]

    def generate(self) -> dict:
        return {
            'bomFormat': 'CycloneDX',
            'specVersion': '1.5',
            'serialNumber': f'urn:uuid:{uuid.uuid4()}',
            'version': 1,
            'metadata': {
                'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
                'tools': [{'vendor': 'Custom', 'name': 'sbom-generator', 'version': '1.0'}],
                'component': {
                    'type': 'application',
                    'name': self.project_name,
                    'version': self.version
                }
            },
            'components': [
                {
                    'type': c.component_type,
                    'name': c.name,
                    'version': c.version,
                    'purl': c.purl,
                    'licenses': [{'license': {'id': lic}} for lic in c.licenses],
                    'hashes': [{'alg': alg, 'content': h} for alg, h in c.hashes.items()],
                    'description': c.description
                }
                for c in self.components
            ]
        }

    def to_json(self, indent: int = 2) -> str:
        return json.dumps(self.generate(), indent=indent)

# Build SBOM for a sample LLM application
gen = CycloneDXGenerator('llm-chat-app', '1.2.0')

PACKAGES = [
    ('langchain', '0.0.247', 'MIT', 'LLM orchestration framework'),
    ('openai', '1.3.0', 'MIT', 'OpenAI Python SDK'),
    ('fastapi', '0.104.0', 'MIT', 'Async web framework'),
    ('pydantic', '2.4.2', 'MIT', 'Data validation'),
    ('aiohttp', '3.9.0', 'Apache-2.0', 'Async HTTP client'),
    ('torch', '2.1.0', 'BSD-3-Clause', 'PyTorch deep learning'),
    ('transformers', '4.35.0', 'Apache-2.0', 'Hugging Face transformers'),
    ('numpy', '1.26.0', 'BSD-3-Clause', 'Numerical computing'),
]

for name, version, license_id, desc in PACKAGES:
    h = hashlib.sha256(f'{name}=={version}'.encode()).hexdigest()[:16]
    gen.add_component(SBOMComponent(
        name=name, version=version,
        purl=f'pkg:pypi/{name}@{version}',
        licenses=[license_id],
        hashes={'SHA-256': h},
        description=desc
    ))

sbom = gen.generate()
print(f'CycloneDX SBOM generated')
print(f'Components: {len(sbom["components"])}')
print(f'Serial: {sbom["serialNumber"]}')
print(f'\nFirst component:')
print(json.dumps(sbom['components'][0], indent=2))


## ML Model SBOM Extension

In [ ]:
@dataclass
class MLModelSBOMComponent(SBOMComponent):
    base_model: Optional[str] = None
    training_dataset: Optional[str] = None
    fine_tuning_method: Optional[str] = None
    model_card_url: Optional[str] = None
    parameter_count: Optional[str] = None

def build_ml_sbom(project: str, version: str) -> dict:
    gen = CycloneDXGenerator(project, version)

    # Add the model itself as a special component
    model = SBOMComponent(
        name='llama-2-7b-chat-finetuned',
        version='1.0.0',
        purl='pkg:huggingface/meta-llama/Llama-2-7b-chat-hf@main',
        licenses=['Llama 2 Community License'],
        hashes={'SHA-256': 'a3f8b9c2d1e4f7a0b5c8d3e6f9a2b5c8'},
        description='Fine-tuned Llama 2 7B for customer support',
        component_type='machine-learning-model'
    )
    gen.add_component(model)

    # Add code dependencies
    for name, version, license_id, desc in PACKAGES[:4]:
        h = hashlib.sha256(f'{name}=={version}'.encode()).hexdigest()[:16]
        gen.add_component(SBOMComponent(
            name=name, version=version,
            purl=f'pkg:pypi/{name}@{version}',
            licenses=[license_id], hashes={'SHA-256': h}, description=desc
        ))

    sbom_data = gen.generate()

    # Extend with ML-specific metadata
    sbom_data['metadata']['mlMetadata'] = {
        'trainingDatasets': [
            {'name': 'alpaca-52k', 'version': '1.0', 'license': 'Apache-2.0'},
            {'name': 'customer-support-internal', 'version': '2023-Q4', 'license': 'Proprietary'}
        ],
        'baseModel': 'meta-llama/Llama-2-7b-hf',
        'fineTuningMethod': 'LoRA',
        'evaluationDataset': 'hellaswag,mmlu',
    }
    return sbom_data

ml_sbom = build_ml_sbom('customer-support-llm', '2.0.0')
print('ML Model SBOM generated')
print(f'Components: {len(ml_sbom["components"])}')
print(f'\nML Metadata:')
print(json.dumps(ml_sbom['metadata']['mlMetadata'], indent=2))


## License Compliance Analysis

In [ ]:
LICENSE_RISK = {
    'GPL-2.0': 'HIGH',    # Copyleft — contaminates proprietary code
    'GPL-3.0': 'HIGH',
    'AGPL-3.0': 'CRITICAL',  # Network copyleft — affects SaaS
    'LGPL-2.1': 'MEDIUM', # Dynamic linking OK, static linking risky
    'MPL-2.0': 'LOW',     # File-level copyleft only
    'Apache-2.0': 'NONE', # Permissive
    'MIT': 'NONE',
    'BSD-3-Clause': 'NONE',
    'BSD-2-Clause': 'NONE',
    'ISC': 'NONE',
    'Llama 2 Community License': 'MEDIUM',  # Commercial restrictions
}

def analyze_licenses(sbom: dict) -> Dict:
    findings = {'CRITICAL': [], 'HIGH': [], 'MEDIUM': [], 'LOW': [], 'UNKNOWN': []}

    for comp in sbom.get('components', []):
        for lic_block in comp.get('licenses', []):
            lic_id = lic_block.get('license', {}).get('id', 'UNKNOWN')
            risk = LICENSE_RISK.get(lic_id, 'UNKNOWN')
            if risk != 'NONE':
                findings[risk].append({'package': comp['name'], 'license': lic_id})

    print('=== License Compliance Report ===')
    for risk in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'UNKNOWN']:
        if findings[risk]:
            print(f'\n{risk}:')
            for f in findings[risk]:
                print(f'  {f["package"]:30s} {f["license"]}')

    return findings

license_findings = analyze_licenses(ml_sbom)
total_issues = sum(len(v) for k, v in license_findings.items() if k != 'UNKNOWN')
print(f'\nTotal license issues: {total_issues}')
